# SmolLM2-135M to Sparse MoE: Fault-Tolerant Upcycling & Fine-Tuning

This notebook demonstrates how to:
1. Mount Google Drive for cross-account persistent checkpointing and shard storage.
2. Clone the repository and install dependencies.
3. Curate the multi-domain Indian contextual dataset into shards.
4. Upcycle `SmolLM2-135M-Instruct` into a 4-expert Top-2 Sparse MoE.
5. Launch parallel dual-process training with automatic session recovery.

## 1. Google Drive Mount & Hardware Verification

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi

## 2. Setup Repository & Dependencies

In [ ]:
import os
if not os.path.exists("/content/linear_MoE_upcycling"):
    !git clone https://github.com/thinktinkergreen/linear_MoE_upcycling.git /content/linear_MoE_upcycling
else:
    !git -C /content/linear_MoE_upcycling pull origin main

%cd /content/linear_MoE_upcycling
!pip install -q -e .  # installs in editable mode once per runtime, registers src globally


## 3. Curate and Shard the Multi-Domain Dataset

In [ ]:
import os
os.environ["PYTHONPATH"] = f"{os.getcwd()}:{os.environ.get("PYTHONPATH", "")}"

!python src/data/curate.py \n  --output_dir /content/drive/MyDrive/linear_MoE_upcycling \n  --num_shards 2


## 4. Run Fault-Tolerant Dual-Worker Training

This runs dual workers with mixed precision (AMP) and atomic checkpoints saving directly to Google Drive. If Colab disconnects, re-running this cell automatically resumes from the latest checkpoint.

In [ ]:
# Pre-Flight Sanity Check: executes 5 steps with strict assertions before full training
!python /content/linear_MoE_upcycling/scripts/sanity_check.py


In [ ]:
import os
os.environ["PYTHONPATH"] = f"{os.getcwd()}:{os.environ.get("PYTHONPATH", "")}"

!python scripts/run_train.py --config configs/default_config.yaml


## 5. Evaluate and Test Generation

In [ ]:
import os
os.environ["PYTHONPATH"] = f"{os.getcwd()}:{os.environ.get("PYTHONPATH", "")}"

!python scripts/evaluate_moe.py \n  --config configs/default_config.yaml \n  --prompt "What are the best methods to grow cotton in Gujarat?"
